# FT-Entmax — fases restantes em DUAS T4 (Kaggle)

Continuação da re-execução do `FTTransformer_entmax` (bisseção corrigida). **O Tier 1 já está pronto e
mesclado no repositório** (`results/tier1_gridcv.json`, 300 registros novos; F1 0,7424, zeros de atenção
0,0798 contra 0,0154 da versão errada) — este notebook não o repete.

**O SAINT não entra aqui.** A versão que rodou antes seguia as Equações 1–2 do artigo (pós-normalização),
enquanto o código oficial usa pré-norma + FFN GEGLU + `dim_head=64` na atenção de linha + FF2 na linha
achatada; a pós-norma colapsa sob lr 1e-3. Será refeito em execução própria com `style="reference"`.

**Fases** (Tier 2, Ablação D, Ablação A, Ablações B/C, Tabela 19):
as caras usam **as duas T4**, um processo por placa com metade das sementes — ≈2× mais rápido e ≈metade
da cota, que conta tempo de sessão. A Tabela 19 roda em uma placa só, por medir tempo e VRAM.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**. Estimativa total: ≈1 h 30 min de sessão.
Tudo é resumível: rodar de novo com o mesmo `--output` continua de onde parou.

**Onde ficam os arquivos.** A célula 2 muda o diretório para o clone, então os runners gravam em
`sparse-lssvm-transformers-study/results/*.json`, **a cada execução concluída** (escrita atômica). O
monitor lê esse mesmo caminho, e a cada atualização de progresso espelha o arquivo em `/kaggle/working`
(raiz do Output), que é de onde se baixa. Se a sessão cair no meio de uma fase, o parcial já está lá:
suba-o como Dataset e aponte `RESUME_DIR` na célula 4 para continuar. Nas fases de duas placas há um
arquivo por GPU (`..._g0.json`, `..._g1.json`), juntados no arquivo da fase ao final.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2 (com 1 placa, use as chamadas run_phase)'

In [ ]:
# ── 2. Repositório (branch com as correções) ──
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')
from src.models.transformers.sparse_attention.entmax_attention import _EntmaxBisectFunction  # noqa
import json, statistics as st
_t1 = [x for x in json.load(open('results/tier1_gridcv.json')) if x['variant'] == 'FTTransformer_entmax' and x['status'] == 'ok']
print(f'entmax corrigido presente; Tier 1 no repo: {len(_t1)} registros, zeros={st.mean(x["mean_zero_fraction"] for x in _t1):.4f}')
assert len(_t1) == 300 and st.mean(x['mean_zero_fraction'] for x in _t1) > 0.05, 'Tier 1 do entmax novo não está no clone — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (todos os parquets vêm no clone; nenhum download) ──
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
from pathlib import Path
falt = [d for d in ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC','ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO'] if not Path(f'data/raw/{d}.parquet').exists()]
assert not falt, f'parquets ausentes: {falt}'
from src.data.loaders import DatasetLoader
for ds in ['TELCO', 'BANK']:
    X, y, _ = DatasetLoader.load(ds); print(f'{ds:<7} N={len(y):>6} p={X.shape[1]}')
print('dados OK')

In [ ]:
# ── 4. Configuração, progresso e paralelismo nas duas placas ──
import shutil, time, subprocess, collections
MODELS = ['FTTransformer_entmax']
MODELS_STR = ' '.join(MODELS)
ABL_A_MODELS = ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax',
                'FTTransformer_sparsemax', 'FTTransformerCURColnorm']   # cinco, sem SAINT
ABL_A_STR = ' '.join(ABL_A_MODELS)
TIER2   = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
SEEDS30 = ' '.join(map(str, range(30)))
SEEDS20 = ' '.join(map(str, range(20)))
OUT = {'tier2': 'results/entmax_tier2.json',  'n5000': 'results/entmax_n5000.json',
       'ablA':  'results/entmax_ablA.json',   'ablBC': 'results/entmax_ablBC.json',
       'scal':  'results/entmax_table19.json'}
nm = len(MODELS)
TOTAL = {'tier2': nm*6*30, 'n5000': nm*6*30, 'ablA': len(ABL_A_MODELS)*3*20, 'ablBC': nm*6*30, 'scal': 1*7}
Path('results').mkdir(exist_ok=True)

RESUME_DIR = None    # ex.: Path('/kaggle/input/entmax-parcial') para retomar sessão anterior
if RESUME_DIR:
    for k, v in OUT.items():
        src = Path(RESUME_DIR) / Path(v).name
        if src.exists(): shutil.copy(src, v); print('restaurado', v)

def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

def _mirror(*paths):
    """Copia os JSONs em curso para /kaggle/working (raiz do Output).

    Os runners gravam em <repo>/results/ a cada execução concluída (append + escrita
    atômica), mas o painel Output do Kaggle mostra /kaggle/working: sem esta cópia
    periódica, uma sessão interrompida no meio de uma fase deixaria os dados só no
    diretório aninhado do clone. Espelhar a cada atualização de progresso torna os
    parciais sempre baixáveis (e reaproveitáveis via RESUME_DIR)."""
    for p in paths:
        try:
            if Path(p).exists(): shutil.copy(p, '/kaggle/working/' + Path(p).name)
        except Exception as e:
            print('  (aviso: falha ao espelhar', p, e, ')')

def _count(path):
    try: r = json.load(open(path))
    except Exception: return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok: return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = f"último: {x.get('dataset', 'N=' + str(x.get('n')))}"
    info += f"/seed{x['seed']}" if 'seed' in x else ''
    info += f" F1={f1:.3f}" if isinstance(f1, (int, float)) else ''
    return len(ok), info

def run_phase(key, cmd, every=60, heartbeat=900):
    """Uma GPU."""
    out, total = OUT[key], TOTAL[key]; log = f'/kaggle/working/{key}.log'
    t0 = time.time(); last_n, last_print = -1, 0.0
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT)
        while True:
            rc = p.poll(); n, info = _count(out); now = time.time()
            if n != last_n or rc is not None or now - last_print > heartbeat:
                el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
                print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
                _mirror(out)                     # parcial sempre baixável
                last_n, last_print = n, now
            if rc is not None: break
            time.sleep(every)
    print(f'[{key}] terminou (exit={rc}) — log: {log}'); save(key)
    if rc != 0: raise RuntimeError(f'{key}: exit {rc}\n' + ''.join(open(log).readlines()[-15:]))

def run_phase_2gpu(key, cmd_fmt, seeds=range(30), every=60, heartbeat=900):
    """Duas T4: um processo por placa, metade das sementes cada, shards separados.

    NÃO usar DataParallel/DDP: o FT-CUR e o SAINT fazem atenção inter-instâncias
    DENTRO do lote, então dividir o lote entre placas mudaria o modelo. Aqui o
    paralelismo é por experimento (sementes), que não altera nada.
    Dois processos gravando o mesmo JSON se sobrescrevem → um arquivo por placa.
    """
    seeds = list(seeds); half = len(seeds)//2
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs, rcs = [], [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        lf = open(log, 'w')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT,
                                      env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min', flush=True)
            _mirror(*shards)                     # parciais sempre baixáveis
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros'); save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs): raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')

print('config OK — modelos:', MODELS)
for k, v in OUT.items(): print(f'  {k:<6} -> {v}  (total {TOTAL[k]})')

In [ ]:
# ── 5. Tier 2 (N=2000) — duas placas ──
run_phase_2gpu('tier2', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 2000 --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 6. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 NOVO) ──
# Moda calculada só do Tier 2 desta execução (não usar extract_tier2_fixed_params.py:
# ele lê todos os results/tier2_transformers*.json, inclusive dumps antigos).
recs = [r for r in json.load(open(OUT['tier2'])) if r.get('status') == 'ok' and r.get('best_params')]
by = collections.defaultdict(list)
for r in recs: by[(r['variant'], r['dataset'])].append(tuple(sorted(r['best_params'].items())))
cfg = json.load(open('config/tier2_fixed_params.json'))
for (v, d), vals in sorted(by.items()):
    mode, cnt = collections.Counter(vals).most_common(1)[0]
    cfg.setdefault(v, {})[d] = dict(mode)
    print(f'{v:<22} {d:<9} {dict(mode)}  [moda {cnt}/{len(vals)}]')
Path('config/tier2_fixed_params_entmax.json').write_text(json.dumps(cfg, indent=2, sort_keys=True))
shutil.copy('config/tier2_fixed_params_entmax.json', '/kaggle/working/tier2_fixed_params_entmax.json')
run_phase_2gpu('n5000', "python -u scripts/run_tier2_fixedparams.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 5000 --config config/tier2_fixed_params_entmax.json"
               " --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 7. Ablação A — CINCO Transformers (sem SAINT) por transferência do Tier 1 ──
# O Tier 1 do entmax já está mesclado em results/tier1_gridcv.json (veio no clone).
# O SAINT será acrescentado quando for refeito (script resumível, chaveado por variante/dataset/semente).
run_phase_2gpu('ablA', "python -u scripts/run_ablation_a_scaling.py --models " + ABL_A_STR +
               " --tier1 results/tier1_gridcv.json --seeds {seeds} --output {output}", seeds=range(20))
recs = json.load(open(OUT['ablA']))
print(collections.Counter((r['variant'], r.get('protocol')) for r in recs if r.get('status') == 'ok'))

In [ ]:
# ── 8. Ablações B + C — duas placas ──
run_phase_2gpu('ablBC', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets TWS_5f TWM_5f TWC_5f MKE MKM MKH --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 9. Tabela 19 — UMA placa (mede tempo e VRAM; nada concorrente) ──
run_phase('scal', f"CUDA_VISIBLE_DEVICES=0 python -u scripts/run_table19_benchmark.py --variants FTTransformer_entmax --repeats 3 --output {OUT['scal']}")

In [ ]:
# ── 10. Resumo ──
for key in ['tier2', 'n5000', 'ablA', 'ablBC']:
    p = Path(OUT[key])
    if not p.exists(): print(key, 'ausente'); continue
    ok = [r for r in json.load(open(p)) if r.get('status') == 'ok']
    f1 = collections.defaultdict(list)
    for r in ok: f1[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
    print(f'\n=== {key}: {len(ok)}/{TOTAL[key]} registros ok ===')
    for (v, d), vals in sorted(f1.items()):
        print(f'  {v:<26} {d:<9} {st.mean(vals):.4f} ± {st.pstdev(vals):.3f}  n={len(vals)}')
print('\nBaixe de /kaggle/working: entmax_{tier2,n5000,ablA,ablBC,table19}.json')